In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import pgmpy
from causallearn.search.ConstraintBased.PC import pc #this seems like a good package for PC

/opt/miniconda3/envs/sklearn-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Credit to Ulrik Thyge Pedersen for the dataset
https://www.kaggle.com/datasets/ulrikthygepedersen/speed-dating/data
adapted from  the 2006 study of Fisman et al
https://doi.org/10.7916/D8FB585Z

In [ ]:
df = pd.read_csv("speeddating.csv")
df = df.drop(columns = ['has_null','wave','match'])
X = df.drop(columns=['decision', 'decision_o'])
labels = df[['decision', 'decision_o']]

Convert ranges b'[a-b]' to the mean value

In [ ]:
def convert_range(x):
    if isinstance(x, str):
        match = re.fullmatch(r"b'\[(\d+)-(\d+)\]'", x)
        if match:
            low, high = map(int, match.groups())
            return (low + high) / 2
    return x

X = X.map(convert_range)

Handling NaNs for PCA/plotting.

In [ ]:
num_cols = X.select_dtypes(include='number').columns
cat_cols = X.select_dtypes(exclude='number').columns

X[num_cols] = SimpleImputer(strategy='median').fit_transform(X[num_cols])

X[cat_cols] = SimpleImputer(strategy='most_frequent').fit_transform(X[cat_cols])

In [ ]:
std_scaler = StandardScaler()
X_encoded = pd.get_dummies(X, drop_first=True)
X_scaled = std_scaler.fit_transform(X_encoded)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

In [ ]:
labels['decision'] = labels['decision'].replace({"b'1'": 1, "b'0'": 0})
labels['decision_o'] = labels['decision_o'].replace({"b'1'": 1, "b'0'": 0})

colors = labels['decision'].astype(int) + labels['decision_o'].astype(int)

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=colors,
    cmap='RdYlGn',
    vmin=0,
    vmax=2,
    s=7,
    alpha=0.4
)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(ticks=[0, 1, 2], label='Number of yes decisions')
plt.show()

Constructing the natural BN. We will condense into 5 meaningful hidden nodes.

In [ ]:
demographic_vars = [
    "has_null",
    "wave",
    "gender",
    "age",
    "age_o",
    "d_age",
    "d_d_age",
    "race",
    "race_o",
    "samerace",
    "importance_same_race",
    "importance_same_religion",
    "d_importance_same_race",
    "d_importance_same_religion",
    "field",
]

interest_vars = [
    "sports", "tvsports", "exercise", "dining",
    "museums", "art", "hiking", "gaming",
    "clubbing", "reading", "tv", "theater",
    "movies", "concerts", "music", "shopping",
    "yoga",

    "d_sports", "d_tvsports", "d_exercise", "d_dining",
    "d_museums", "d_art", "d_hiking", "d_gaming",
    "d_clubbing", "d_reading", "d_tv", "d_theater",
    "d_movies", "d_concerts", "d_music", "d_shopping",
    "d_yoga",

    "interests_correlate",
    "d_interests_correlate",
]

preference_vars = [
    "attractive_important",
    "sincere_important",
    "intellicence_important",
    "funny_important",
    "ambtition_important",
    "shared_interests_important",

    "d_attractive_important",
    "d_sincere_important",
    "d_intellicence_important",
    "d_funny_important",
    "d_ambtition_important",
    "d_shared_interests_important",

    "importance_same_race",
    "importance_same_religion",
    "d_importance_same_race",
    "d_importance_same_religion",
]



TODO: Run MLE on our BN to construct CPTs (use only ~85% of the data, reserve 15% for testing)
- If we decide to introduce hidden nodes, we will instead need to use EM to construct the CPTs.

TODO: Use the PC algorithm to construct a a BN, use MLE again to construct CPTs

TODO: Use PC algorithm again, this given only survey responses. How does it compare?

TODO: Is there any nice data visualization we can do?